# TP 02 | À partir de quel score déclencher une inspection ? (corrigé)

**Chapitre 2. Évaluer un modèle**

Chez Nervure Composites, Camille dispose de 20 inspections renforcées pour 100 lots. À la sortie de la presse, elle connaît la température, la pression, la durée et la ligne. Le laboratoire rend son verdict huit heures plus tard. Dans ce notebook, vous allez **entraîner deux modèles**, vérifier leurs prédictions sur des lots ultérieurs, puis remettre une règle de priorité à Camille. L’équipe qualité garde la main sur la libération des lots.

**Pour démarrer.** Depuis le dossier du projet, lancez `uv sync --locked --extra spark` pour installer les bibliothèques. Ouvrez ce notebook dans VS Code, choisissez le noyau de `.venv` et exécutez les cellules dans l’ordre avec Maj + Entrée. Les données synthétiques sont fournies. Les blocs « À vous » proposent un calcul ou une décision à compléter. Conservez dans votre notebook la recommandation finale, puis choisissez un défi facultatif.

**Parcours.** Vérifiez la chronologie, entraînez les modèles, interprétez leurs erreurs puis remettez une règle sous contrainte de capacité. Les quatre défis suivants sont indépendants : coûts, quota par période, calibration et désaccords.

## 1. Quels résultats aurait-on réellement connus à la sortie de la presse ?

**Atelier 2.1 du support MLB.** Sections 1 à 2.

On confie **720 lots anciens** à l’entraînement et **240 lots ultérieurs** à la validation. L’entraînement sert à ajuster les paramètres ; la validation sert à comparer les modèles et à choisir le seuil. Le test final du TP 08 est réservé à l’évaluation de la règle après ces choix. Utiliser ce dernier pour régler un seuil rendrait sa mesure de performance trop optimiste.

![Trois périodes successives : ajuster sur le passé, choisir sur la validation puis évaluer sur le test réservé.](../figures/decoupage.svg)

**À vous.** Avant d’exécuter, choisissez les deux dates à comparer. Une comparaison entre les seules dates de production suffirait-elle si certains résultats du laboratoire arrivaient en retard ?

In [ ]:
# Charger les deux périodes et vérifier que les verdicts anciens précèdent la validation.
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

racine = Path.cwd()
if racine.name == "corrections":
    racine = racine.parent


def lire(nom):
    return pd.read_csv(
        racine / "donnees" / nom, parse_dates=["date", "label_disponible"]
    )


train = lire("lots_train.csv")
val = lire("lots_validation.csv")
for jeu in [train, val]:
    assert jeu["id"].is_unique and jeu["date"].is_monotonic_increasing
assert set(train["id"]).isdisjoint(val["id"])
assert train["label_disponible"].max() < val["date"].min()
print("Dernier verdict du train :", train["label_disponible"].max())
print("Première décision de validation :", val["date"].min())

**Corrigé.** Le dernier verdict d’entraînement doit être connu avant la première décision simulée en validation. La date de production indique quand on mesure un lot, pas quand son étiquette peut servir à apprendre. Un même lot dans les deux jeux fausserait également la comparaison.

**Les deux périodes se ressemblent-elles ?** La représentativité se juge par rapport aux futurs lots que Camille devra traiter. Comparez la fréquence des défauts, la présence de la presse B et les pressions absentes. Un écart mérite une enquête ; ce tableau seul ne permet pas d’en attribuer la cause. Si plusieurs lignes du fichier décrivaient le même panneau, on garderait toutes ses mesures dans le même groupe lors du découpage.

In [ ]:
# Comparer les périodes en proportions pour tenir compte de leurs effectifs différents.
def profil(jeu):
    return pd.Series(
        {
            "lots": len(jeu),
            "défauts (%)": 100 * jeu["defaut"].mean(),
            "presse B (%)": 100 * jeu["ligne"].eq("B").mean(),
            "pression absente (%)": 100 * jeu["pression"].isna().mean(),
        }
    )


display(pd.DataFrame({"train": profil(train), "validation": profil(val)}).round(1))
# À vous : quelle proportion varie le plus en points de pourcentage ?

**Corrigé.** La fréquence des défauts passe de 26,7 % à 32,5 %, soit environ 5,8 points de plus. Comparez cet écart aux deux autres lignes du tableau. La présence des deux presses ne garantit pas que les conditions de production resteront stables.

## 2. Apprendre un score sur les lots anciens

La **régression logistique** est ici un classifieur. Elle combine les mesures par une somme pondérée puis transforme cette somme en un score compris entre 0 et 1. Pendant `fit`, elle ajuste ses coefficients d’après les verdicts. Un **arbre de décision** apprend une succession de questions sur les entrées ; son score dépend de la proportion de défauts dans la feuille atteinte. Un score de 0,7 indique un risque estimé par le modèle. Pour l’interpréter comme « environ 7 défauts sur 10 », il faudrait vérifier cette concordance sur d’autres données.

![Courbe en S reliant la somme pondérée des mesures au score logistique](../figures/concept-logistique.svg)

Une somme de zéro donne un score de 0,5. La courbe transforme la somme en score entre 0 et 1 ; Camille choisira séparément le seuil d’inspection.

Dans les deux cas, on remplace les mesures manquantes par les médianes du train, on centre et réduit les mesures numériques et on représente la ligne A ou B par des indicateurs. Le *pipeline* enchaîne ces opérations et le classifieur : un seul appel à `fit` apprend toute la chaîne. On conserve L0010 dans cette comparaison pour que les modèles voient exactement les mêmes lots. Sa mesure signalée au TP 01 reste à expertiser.

In [ ]:
# Ajuster les deux modèles sur les mêmes mesures pour comparer leurs scores sur les lots suivants.
mesures = ["temperature", "pression", "duree"]
entrees = mesures + ["ligne"]


def chaine(classifieur):
    numerique = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())
    preparation = ColumnTransformer(
        [
            ("mesures", numerique, mesures),
            ("presse", OneHotEncoder(handle_unknown="ignore"), ["ligne"]),
        ]
    )
    return make_pipeline(preparation, classifieur)


modeles = {
    "Logistique": chaine(LogisticRegression(max_iter=1000)),
    "Arbre": chaine(
        DecisionTreeClassifier(max_depth=3, min_samples_leaf=12, random_state=42)
    ),
}
# Appliquer chaque modèle aux mêmes lots de validation.
scores = {}
for nom, modele in modeles.items():
    modele.fit(train[entrees], train["defaut"])
    scores[nom] = modele.predict_proba(val[entrees])[:, 1]

display(pd.DataFrame(scores, index=val["id"]).head().round(3))
mediane_apprise = (
    modeles["Logistique"][0].named_transformers_["mesures"][0].statistics_[1]
)
print("Pression médiane apprise :", round(mediane_apprise, 4))
assert np.isclose(mediane_apprise, train["pression"].median())

**À vous.** Identifiez dans la cellule précédente l’instruction qui modifie les paramètres et celle qui produit les scores. Pourquoi la colonne `controle_final` ne figure-t-elle jamais dans `entrees` ? Pour voir l’effet de la préparation sur un cas concret, affichez une ligne de validation dont la pression est absente et retrouvez son score.

**Manipulation guidée.** Sur le premier lot de validation, recherchez ce que prédit chaque modèle et appliquez le seuil de 0,50. Recommencez sur le deuxième lot. Pour le deuxième, les deux modèles aboutissent-ils à la même décision au seuil de 0,80 ?

In [ ]:
# Suivre un dossier incomplet puis observer comment un seuil transforme un score en décision.
incomplets = val["pression"].isna()
position = np.flatnonzero(incomplets)[0]
display(val.iloc[[position]][["id", "pression", "ligne"]])
print("Score logistique :", round(scores["Logistique"][position], 3))
for position in [0, 1]:
    for nom, valeurs in scores.items():
        score = valeurs[position]
        print(
            val.iloc[position]["id"], nom, round(score, 3), score >= 0.50, score >= 0.80
        )

**Corrigé.** L’ajustement se fait uniquement avec fit sur le train. predict_proba applique ensuite la préparation déjà apprise aux mesures de validation. Une pression absente peut donc recevoir un score. Le contrôle final révélerait la cible après la décision : même une excellente performance obtenue avec cette colonne serait inutilisable au poste. Sur L0720, les deux règles laissent le lot en file ordinaire à 0,50. Sur L0721, les deux signalent le lot à 0,50 ; à 0,80, seul l’arbre le signale (0,936 contre 0,671).

### Une comparaison tient-elle sur trois campagnes successives ?

![Trois historiques croissants évalués chacun sur les 120 lots suivants.](../figures/tp02-trois-fenetres.svg)

Un découpage chronologique à fenêtre croissante rejoue trois dates de décision. À chaque date, on réapprend les deux pipelines avec les seuls verdicts déjà arrivés. On compare ensuite leurs classements sur les 120 lots suivants. Une AUC de 0,5 correspond au classement attendu par hasard. Quel modèle gagne à chaque passage ? Comparez ensuite les écarts entre périodes avec les écarts entre modèles.

In [ ]:
# Rejouer trois validations à l’intérieur de l’historique, en préservant les modèles déjà ajustés.
from sklearn.base import clone

rejeux = []
for passage, (avant, apres) in enumerate(
    TimeSeriesSplit(n_splits=3, test_size=120).split(train), 1
):
    campagne = train.iloc[apres]
    anciens = train.iloc[avant]
    connus = anciens[anciens.label_disponible < campagne.date.min()]
    for nom, modele in modeles.items():
        copie = clone(modele).fit(connus[entrees], connus.defaut)
        risque = copie.predict_proba(campagne[entrees])[:, 1]
        rejeux.append(
            (passage, len(connus), nom, roc_auc_score(campagne.defaut, risque))
        )
bilan_fenetres = pd.DataFrame(
    rejeux, columns=["passage", "lots connus", "modèle", "AUC"]
)
display(
    bilan_fenetres.pivot(
        index=["passage", "lots connus"], columns="modèle", values="AUC"
    ).round(3)
)

**Corrigé.** La logistique obtient 0,916, 0,961 et 0,856, contre 0,891, 0,913 et 0,816 pour l’arbre. L’écart entre la meilleure et la moins bonne campagne de la logistique (0,105) dépasse son avance sur l’arbre dans chaque campagne. Dans chaque ligne, la plus grande AUC désigne le meilleur classement de cette campagne. Comparez ensuite les valeurs d’une même colonne pour voir la variabilité du modèle selon la période. Ces trois mesures partagent une partie de leur historique d’apprentissage. Leurs écarts décrivent la stabilité observée, sans constituer trois expériences indépendantes. Le test final du TP 08 reste intact.

## 3. Lire une décision : où vont les erreurs ?

**Atelier 2.2 du support MLB.** Sections 3 à 4.

Prenez d’abord **100 lots fictifs**, indépendants du CSV. Dix présentent un défaut. Au seuil de 0,50, on signale 20 lots et 8 d’entre eux sont effectivement défectueux. Complétez la **matrice de confusion** : les lignes sont le verdict réel, les colonnes la décision. Un vrai positif (VP) est un défaut signalé ; un faux positif (FP) une fausse alerte ; un faux négatif (FN) un défaut manqué ; un vrai négatif (VN) un lot conforme non signalé.

![Matrice à compléter avec les totaux du scénario de 100 lots](../figures/tp02-lire-matrice.svg)

**À vous.** Retrouvez les quatre cases et calculez les trois indicateurs.

$$
\begin{aligned}
\text{Précision} &= \frac{\mathrm{VP}}{\mathrm{VP}+\mathrm{FP}} \\[5pt]
\text{Rappel} &= \frac{\mathrm{VP}}{\mathrm{VP}+\mathrm{FN}} \\[5pt]
\text{Exactitude} &= \frac{\mathrm{VP}+\mathrm{VN}}{\text{nombre total de lots}}
\end{aligned}
$$

Quelle fraction intéresse Camille pour juger l’usage de ses créneaux ? Et pour mesurer les défauts qu’elle intercepte ?

In [ ]:
# Reconstituer les quatre issues possibles à partir des inspections et des verdicts connus.
vp = 8
fp = 20 - vp
fn = 10 - vp
vn = 100 - vp - fp - fn
print(
    f"Précision {vp / (vp + fp):.0%}, rappel {vp / (vp + fn):.0%}, exactitude {(vp + vn) / 100:.0%}"
)
assert (vn, fp, fn, vp) == (78, 12, 2, 8)

**Corrigé.** Huit des vingt inspections interceptent un défaut, sur dix défauts réels.

$$
\begin{aligned}
\text{Précision} &= \frac{8}{20} = 40\,\% \\[5pt]
\text{Rappel} &= \frac{8}{10} = 80\,\% \\[5pt]
\text{Exactitude} &= \frac{8+78}{100} = 86\,\%
\end{aligned}
$$

Ne rien inspecter aurait 90 % d’exactitude mais manquerait les dix défauts. Sa précision serait indéfinie puisqu’aucun lot ne serait signalé.

### Passer des scores aux décisions sur les 240 lots

La règle est inclusive : un score **égal** au seuil déclenche une inspection. La fonction suivante applique la même convention à tous les modèles. Lisez les deux matrices de la logistique. En abaissant le seuil, quelles cases diminuent ? Combien de créneaux supplémentaires sont nécessaires ?

In [ ]:
# Comparer les décisions aux verdicts pour voir quelles erreurs changent entre deux seuils.
y = val["defaut"].to_numpy()


def bilan(scores_lots, seuil):
    decision = scores_lots >= seuil
    vn, fp, fn, vp = confusion_matrix(y, decision, labels=[0, 1]).ravel()
    return {"VN": vn, "FP": fp, "FN": fn, "VP": vp, "inspections": fp + vp}


fig, axes = plt.subplots(1, 2, figsize=(8, 3.5), layout="constrained")
for ax, seuil in zip(axes, [0.50, 0.30], strict=True):
    b = bilan(scores["Logistique"], seuil)
    matrice = np.array([[b["VN"], b["FP"]], [b["FN"], b["VP"]]])
    ax.imshow(matrice, cmap="Blues", vmin=0, vmax=len(val))
    for (i, j), valeur in np.ndenumerate(matrice):
        ax.text(j, i, str(valeur), ha="center", va="center", fontsize=15)
    ax.set(
        xticks=[0, 1],
        xticklabels=["Ordinaire", "Prioritaire"],
        yticks=[0, 1],
        yticklabels=["Conforme", "Défaut"],
        ylabel="Verdict du laboratoire",
        xlabel="Décision",
        title=f"Seuil {seuil:.2f}",
    )
plt.show()

### ROC et précision-rappel : lire tous les seuils à la fois

Sur la **courbe ROC** (*receiver operating characteristic*), l’axe horizontal indique la proportion de lots conformes signalés à tort.

![Quatre scores classés et aire sous la courbe ROC correspondante](../figures/concept-roc.svg)

Au seuil inclusif de 0,5, A, B et C sont signalés : les deux défauts sont détectés et un conforme sur deux déclenche une fausse alerte. Le point orange correspond à 50 % de fausses alertes et 100 % de rappel. Sur les quatre paires possibles associant un défaut à un conforme, trois sont dans le bon ordre. On retrouve l’AUC de 0,75.

$$
\text{Taux de fausses alertes} = \frac{\mathrm{FP}}{\mathrm{FP}+\mathrm{VN}}
$$

L’axe vertical donne le rappel. L’**AUC ROC**, aire sous cette courbe, mesure la capacité à classer un lot défectueux devant un conforme, en comptant une égalité pour moitié. Sur la courbe **précision-rappel**, on lit directement le compromis entre la part de défauts détectés et la part d’inspections utiles. La ligne de référence horizontale correspond à la fréquence de défaut dans la période.

Repérez les points 0,50 et 0,30 sur les deux courbes de la logistique. Changer le seuil déplace le point de fonctionnement sur la même courbe. Le coût et le nombre de places vont déterminer lequel retenir.

Pour vérifier votre lecture, imaginez que Camille inspecte les quatre lots de la figure. Elle trouve les deux défauts. Son rappel vaut 2/2, sa précision vaut 2/4 et elle mobilise quatre créneaux. Le classement des scores et son AUC sont pourtant identiques à ceux de la figure. Pour choisir un seuil, il reste donc à compter les places consommées à ce seuil.

In [ ]:
# Comparer les classements sur tous les seuils et situer les deux seuils étudiés sur les courbes.
fig, (roc, pr) = plt.subplots(1, 2, figsize=(9, 3.8), layout="constrained")
for nom, valeurs in scores.items():
    fpr, tpr, _ = roc_curve(y, valeurs)
    precision, rappel, _ = precision_recall_curve(y, valeurs)
    roc.plot(fpr, tpr, label=f"{nom} (AUC {roc_auc_score(y, valeurs):.2f})")
    pr.step(rappel, precision, where="post", label=nom)
for seuil, marqueur in [(0.50, "o"), (0.30, "s")]:
    b = bilan(scores["Logistique"], seuil)
    rappel = b["VP"] / (b["VP"] + b["FN"])
    for ax, x, z in [
        (roc, b["FP"] / (b["FP"] + b["VN"]), rappel),
        (pr, rappel, b["VP"] / b["inspections"]),
    ]:
        ax.scatter(
            x, z, marker=marqueur, color="black", label=f"Seuil {seuil:.2f}", zorder=4
        )
roc.plot([0, 1], [0, 1], ":", color="gray")
pr.axhline(y.mean(), linestyle=":", color="gray")
roc.set(xlabel="Taux de fausses alertes", ylabel="Rappel", title="ROC")
pr.set(xlabel="Rappel", ylabel="Précision", title="Précision-rappel")
for ax in (roc, pr):
    ax.set(xlim=(-0.02, 1.02), ylim=(-0.03, 1.03))
    ax.legend(fontsize=8)
plt.show()

## 4. Choisir une règle compatible avec l’atelier

Camille accorde **1 point de coût** à une fausse alerte et **10 points** à un défaut manqué. Les bonnes décisions valent zéro dans ce barème simplifié. Chaque lot signalé consomme une place, quel que soit son verdict.

$$
\begin{aligned}
\text{Coût total} &= \mathrm{FP}+10\times\mathrm{FN} \\[5pt]
\text{Inspections} &= \mathrm{VP}+\mathrm{FP} \\[5pt]
\text{Capacité maximale} &= \frac{20}{100}\times 240 = 48
\end{aligned}
$$

Ce budget global sert à comparer les règles ; en exploitation il faudra appliquer la capacité à chaque file réelle.

Comparez une référence « aucune priorité », l’arbre et la logistique. On essaie des seuils de 0,05 à 1,00 par pas de 0,05. Parmi les règles admissibles, choisissez le coût le plus bas. En cas d’égalité, préférez le moins d’inspections, puis le seuil le plus élevé pour un même modèle. Un seuil choisi ici reste à confirmer sur un test indépendant.

In [ ]:
# Écarter les règles qui dépassent la capacité avant de comparer le coût des erreurs.
capacite = len(val) * 20 // 100
reference = {
    "modèle": "Aucune priorité",
    "seuil": np.nan,
    **bilan(np.zeros(len(val)), 1.0),
}
lignes = [reference]
for nom, valeurs in scores.items():
    for centiemes in range(5, 101, 5):
        seuil = centiemes / 100
        lignes.append({"modèle": nom, "seuil": seuil, **bilan(valeurs, seuil)})
essais = pd.DataFrame(lignes)
essais["coût"] = essais["FP"] + 10 * essais["FN"]
# Filtrer sur les places disponibles avant de départager les règles.
admissibles = essais.loc[essais["inspections"] <= capacite].copy()
ordre = ["coût", "inspections", "seuil"]
classement = admissibles.sort_values(ordre, ascending=[True, True, False])
display(classement.drop_duplicates("modèle").reset_index(drop=True))
print("Capacité :", capacite)

### Sur quelle presse manque-t-on des défauts ?

Le laboratoire veut savoir où enquêter en priorité. Gardez la règle en tête du classement, puis calculez le **rappel par presse**. Son dénominateur est le nombre de défauts réels de la presse. Une moyenne calculée sur tous les lots répondrait à une autre question.

1. Complétez le journal avec un booléen « défaut intercepté ». 2. Additionnez les défauts et les interceptions par ligne. 3. Calculez les défauts manqués et le rappel. Comparez la priorité suggérée par le nombre de défauts réels à celle suggérée par le nombre de défauts manqués.

In [ ]:
# Chercher sur quelle presse les défauts échappent le plus souvent à la règle retenue.
regle = classement.iloc[0]
journal = val[["id", "date", "ligne", "defaut"]].copy()
journal["score"] = scores[regle["modèle"]]
journal["alerte"] = journal["score"] >= regle["seuil"]
display(journal.head())
# Combinez alerte et defaut.eq(1) avec &. Regroupez ensuite par ligne.
audit = journal.assign(intercepte=journal["alerte"] & journal["defaut"].eq(1))
presse = audit.groupby("ligne")[["defaut", "intercepte"]].sum()
presse["manque"] = presse["defaut"] - presse["intercepte"]
presse["rappel"] = presse["intercepte"] / presse["defaut"]
display(presse)
assert presse["manque"].sum() == regle["FN"]

**Lecture du bilan.** Sur A, on intercepte 9 défauts sur 31 (29 %) et on en manque 22. Sur B, on en intercepte 31 sur 47 (66 %) et on en manque 16. B compte davantage de défauts, mais A concentre plus de défauts manqués. J’examinerais les dossiers manqués sur A avant de proposer un changement de seuil. La comparaison décrit cette période de validation.

### Le planning peut-il absorber les alertes à chaque période ?

![Deux plannings de huit alertes réparties entre deux périodes.](../figures/tp02-planning.svg)

Dans cet exemple miniature, chaque période offre quatre places. Prévoyez lequel des deux plannings oblige à reporter des inspections, et combien. Appliquez ensuite le même raisonnement aux six périodes de Nervure.

Les créneaux inutilisés ne sont pas forcément reportables. Pour cette simulation, le responsable répartit les 240 lots en **six périodes consécutives de 40**, avec huit inspections possibles dans chacune. Prévoyez si le respect du budget global garantit le respect de chaque budget local.

Triez le journal par date. Le code numérote les périodes de 1 à 6 avec une division entière. Calculez le nombre d’alertes par période, tracez les six valeurs et une ligne horizontale à huit. Chiffrez les inspections en excès en comptant uniquement les dépassements.

In [ ]:
# Vérifier la charge dans chaque période car les créneaux inutilisés ne sont pas reportables.
planning = journal.sort_values(["date", "id"]).reset_index(drop=True)
planning["periode"] = planning.index // 40 + 1
budget_local = 8
# Additionnez alerte par période. Utilisez plot.bar() puis axhline().
# Pour les dépassements, soustrayez le budget et ramenez les valeurs négatives à zéro.
charge = planning.groupby("periode")["alerte"].sum()
ax = charge.plot.bar(
    color="#277568", rot=0, ylabel="Inspections demandées", xlabel="Période de 40 lots"
)
ax.axhline(budget_local, linestyle="--", color="#a43c1e", label="8 places")
ax.legend()
plt.show()
exces = (charge - budget_local).clip(lower=0)
display(pd.DataFrame({"alertes": charge, "excès": exces}))
assert exces.sum() == 3

**Décision de planning.** Dans le planning miniature, A utilise quatre places à chaque période. B en demande sept en période 2 : trois alertes dépassent la capacité locale, malgré le même total de huit. Sur les données de validation, la règle demande successivement 2, 9, 8, 8, 7 et 10 inspections. Les 44 alertes tiennent dans les 48 places globales, mais il manque une place en période 2 et deux en période 6. Le responsable doit préciser si l’on peut renforcer ces périodes ou comment arbitrer dès que les huit créneaux ont été utilisés. Cette contrainte change la règle effectivement appliquée.

**À remettre à Camille.** Indiquez la période et le nombre de lots utilisés, le modèle, le seuil, la charge, les défauts interceptés et manqués ainsi que le coût. Comparez à la référence. Quelle décision prendre si demain le seuil signale plus de 20 lots sur 100 ? Quelle mesure gardez-vous pour le test final ?

In [ ]:
# Chiffrer la règle retenue et le compromis entre inspections et défauts manqués.
choix = classement.iloc[0]
print(choix.to_string())
b50 = bilan(scores["Logistique"], 0.5)
b30 = bilan(scores["Logistique"], 0.3)
print("Inspections supplémentaires :", b30["inspections"] - b50["inspections"])
print("Défauts manqués en moins :", b50["FN"] - b30["FN"])

**Corrigé.** Sur les 240 lots suivant les 720 lots anciens, je retiens la logistique avec un seuil inclusif de 0,70. Elle déclenche 44 inspections sur les 48 possibles, dont 40 utiles et 4 fausses alertes. Elle manque 38 défauts sur 78. On obtient le coût suivant.

$$
\begin{aligned}
C &= 4 + 10\times 38 \\[5pt]
&= 384\ \text{points}
\end{aligned}
$$

Sans inspection, le coût atteint 780 points. La meilleure règle admissible de l’arbre sur cette grille en coûte 532. Le coût économisé par rapport à l’absence d’alerte ne vaut que pour ce barème. Si une file réelle dépasse la capacité, classer les candidats par score, départager les ex æquo selon une règle convenue et suivre les lots en attente. Évaluer ensuite cette politique complète sur les lots réservés, sans réajuster le seuil.

## Défi facultatif | Une recommandation résiste-t-elle à un changement d’hypothèse ?

1. Reprenez les deux matrices sur les **100 lots fictifs**. Au seuil de 0,30, il y a maintenant 9 vrais positifs et 20 faux positifs. Calculez le coût des deux seuils si un défaut manqué coûte 10 points, puis 2. Distinguez le choix économique de la faisabilité.
2. Sur les **240 lots industriels**, modifiez le coût du défaut manqué à 2. Refaites le classement sous la même contrainte de capacité. Le modèle ou le seuil change-t-il ?
3. Pour répéter l’évaluation sur plusieurs périodes, découpez les 720 lots d’entraînement en trois fenêtres successives avec TimeSeriesSplit. Pour chaque fenêtre, vérifiez que le dernier verdict d’entraînement arrive avant la première décision suivante. Expliquez comment ajuster le pipeline dans ces fenêtres.

In [ ]:
# Tester si la recommandation dépend du coût d’un défaut manqué et du découpage temporel.
for cout_fn in [10, 2]:
    print(
        "Coût du défaut",
        cout_fn,
        "| seuil 0,50 :",
        12 + cout_fn * 2,
        "| seuil 0,30 :",
        20 + cout_fn,
    )
autre = admissibles.assign(coût=admissibles["FP"] + 2 * admissibles["FN"])
display(autre.sort_values(ordre, ascending=[True, True, False]).head(1))
# Respecter la chronologie des verdicts dans chaque fenêtre.
for passe, suite in TimeSeriesSplit(n_splits=3).split(train):
    assert train.iloc[passe]["label_disponible"].max() < train.iloc[suite]["date"].min()
    print(len(passe), "lots pour apprendre puis", len(suite), "pour vérifier")

**Corrigé.** Sur 100 lots, les coûts à 0,50 et 0,30 valent 32 et 30 avec la pénalité de 10, puis 16 et 22 avec la pénalité de 2. Même quand 0,30 est moins cher, ses 29 alertes dépassent les 20 places. Sur les lots industriels, la logistique reste en tête mais le seuil monte à 0,75. On inspecte 41 lots pour 2 fausses alertes et 39 défauts manqués : 80 points avec le nouveau barème. Comparer ce total directement aux 384 points de l’ancien barème serait trompeur. Les trois fenêtres utilisent 180, 360 et 540 lots du passé pour vérifier les 180 suivants. À chaque fenêtre on réajuste les médianes et le modèle exclusivement sur son passé. Mélanger les dates avant le découpage rendrait ce contrôle inopérant.

## Défi facultatif | Simuler une règle exécutable sans connaître les lots suivants

Camille veut tester une réponse concrète aux dépassements. Conservez le seuil retenu. Dans chaque période, inspectez les huit **premières alertes en ordre chronologique**, puis arrêtez. Avant le calcul, expliquez pourquoi choisir les huit meilleurs scores de toute la période supposerait de connaître les lots futurs.

Utilisez la somme cumulée des alertes par période comme compteur. Recalculez les vrais positifs, les fausses alertes, les défauts manqués et le coût à 10 points par défaut manqué. Combien de places et combien de points séparent les deux politiques ?

In [ ]:
# Simuler l’attribution des places au fil des arrivées avec un quota remis à zéro à chaque période.
rang_alerte = planning.groupby("periode")["alerte"].cumsum()
# Un lot est inspecté si son score déclenche une alerte ET si le compteur est au plus huit.
# Comparez les deux décisions au même verdict, sur les mêmes lots.
inspecte = planning["alerte"] & (rang_alerte <= budget_local)
vn, fp, fn, vp = confusion_matrix(planning["defaut"], inspecte, labels=[0, 1]).ravel()
cout_plafonne = fp + 10 * fn
print(
    f"{inspecte.sum()} inspections, {vp} défauts interceptés, {fp} fausses alertes, {fn} défauts manqués."
)
print(f"Écart de coût : {cout_plafonne - regle['coût']:+.0f} points")
display(
    planning.loc[planning["alerte"] & ~inspecte, ["id", "periode", "score", "defaut"]]
)
assert inspecte.groupby(planning["periode"]).sum().max() <= budget_local

**Interprétation.** Le compteur n’utilise que les alertes déjà reçues. La contrainte retire trois inspections, ramenant la charge de 44 à 41. Les trois alertes écartées correspondent à des défauts. On intercepte 37 défauts au lieu de 40, avec toujours quatre fausses alertes. Le coût passe de 384 à 414 points. Le respect du planning coûte donc 30 points sur cette période. Les verdicts servent uniquement à évaluer la politique après coup. Un classement des huit meilleurs scores deviendrait envisageable si l’atelier pouvait attendre la fin de la période avant toute décision.

## Défi facultatif | Un score de 0,88 correspond-il vraiment à 88 % de défauts ?

Un modèle est **calibré** si les lots auxquels il attribue une probabilité proche de 0,8 présentent un défaut environ huit fois sur dix. On examine ici des groupes de lots car un verdict individuel vaut uniquement 0 ou 1.

1. Répartissez les scores logistiques de validation en quatre tranches. Pour chacune, calculez l’effectif, le score moyen et la fréquence observée des défauts.
2. Tracez la fréquence observée en fonction du score moyen, puis la diagonale de référence. Sur quel côté de la diagonale le risque est-il sous-estimé ?
3. Comparez la tranche la plus basse et la plus haute. Pourquoi faut-il conserver les effectifs à côté du graphique ?

In [ ]:
# Comparer les scores moyens aux fréquences observées pour juger la calibration du modèle.
calibration = pd.DataFrame(
    {"score": scores["Logistique"], "defaut": val["defaut"].to_numpy()}
)
calibration["tranche"] = pd.cut(
    calibration["score"], [0, 0.25, 0.50, 0.75, 1], include_lowest=True
)
# Groupez par tranche. Utilisez size pour compter les lots et mean sur les deux colonnes.
# Pour la diagonale, reliez les points (0, 0) et (1, 1).
fiabilite = calibration.groupby("tranche", observed=True).agg(
    lots=("score", "size"), annonce=("score", "mean"), observe=("defaut", "mean")
)
display(fiabilite.round(3))
ax = fiabilite.plot.scatter(x="annonce", y="observe", color="#26756a")
ax.plot([0, 1], [0, 1], linestyle="--", color="gray")
ax.set(
    xlim=(0, 1),
    ylim=(0, 1),
    xlabel="Score moyen annoncé",
    ylabel="Fréquence des défauts",
)
plt.show()
assert fiabilite["lots"].sum() == len(val)

**Correction.** Au-dessus de la diagonale, il y a davantage de défauts que prévu. La première tranche regroupe 147 lots avec un score moyen de 6,5 % pour 12,9 % de défauts observés. Dans la dernière, 41 lots ont un score moyen de 88,2 % et 95,1 % présentent un défaut. Une seule observation change la fréquence de cette dernière tranche d’environ 2,4 points à effectif constant. Ces écarts sur la validation méritent un suivi sur d’autres périodes. Ils ne suffisent pas à promettre une probabilité individuelle exacte.

## Défi facultatif | Quels lots les deux modèles inspecteraient-ils différemment ?

Camille veut comprendre le désaccord entre l’arbre et la logistique. Faites une **comparaison appariée** : les deux modèles prennent une décision sur chacun des mêmes lots. Pour cette enquête, utilisez un seuil inclusif de 0,50 pour chacun. Ce seuil commun est un scénario de diagnostic indépendant du seuil opérationnel retenu.

1. Formez les quatre groupes possibles (aucune alerte, logistique seule, arbre seul, deux alertes).
2. Pour chacun, comptez les lots et les défauts vérifiés. Combien d’inspections supplémentaires et de défauts interceptés obtiendrait-on en suivant toute alerte de l’un ou de l’autre modèle, par rapport à la logistique seule ?
3. Repérez un lot sur lequel les modèles divergent. Quand le verdict peut-il servir à arbitrer ? La règle combinée respecte-t-elle les 48 places sur la validation ?

In [ ]:
# Suivre les deux modèles lot par lot pour mesurer le gain et la charge de leurs alertes combinées.
avis = val[["id", "defaut"]].copy()
avis["logistique"] = scores["Logistique"] >= 0.50
avis["arbre"] = scores["Arbre"] >= 0.50
# Groupez par les deux décisions et utilisez size et sum sur defaut.
# Le symbole | combine deux colonnes booléennes par un « ou ».
croise = avis.groupby(["logistique", "arbre"])["defaut"].agg(["size", "sum"])
display(croise)
union = avis["logistique"] | avis["arbre"]
ajouts = union & ~avis["logistique"]
display(avis.loc[ajouts].head(1))
print("Inspections combinées :", union.sum())
print(
    "Inspections ajoutées :",
    ajouts.sum(),
    "dont défauts :",
    avis.loc[ajouts, "defaut"].sum(),
)
assert union.sum() == 81

**Correction.** Les deux modèles signalent ensemble 59 lots, dont 48 défectueux. La logistique seule en signale quatre autres, dont deux défectueux. L’arbre seul en signale 18, dont quatre défectueux. Les 159 lots restants incluent 24 défauts. Passer de la logistique à l’union ajoute donc 18 inspections pour gagner quatre détections et produire 14 fausses alertes supplémentaires. Avec 81 inspections pour 48 places, cette règle est irréalisable dans le scénario. À la sortie de presse, le désaccord peut orienter une revue. Le verdict du laboratoire servira à l’évaluer huit heures plus tard.

### Pour aller plus loin | Évaluer une cible numérique

 Pour une cible numérique, la matrice de confusion perd son sens. Avec deux écarts de prédiction de 1 et 4 MPa, on peut calculer l’erreur absolue moyenne (MAE) et la racine de l’erreur quadratique moyenne (RMSE).

$$
\begin{aligned}
\mathrm{MAE} &= \frac{|1|+|4|}{2} \\[5pt]
&= 2{,}5\ \mathrm{MPa} \\[5pt]
\mathrm{RMSE} &= \sqrt{\frac{1^2+4^2}{2}} \\[5pt]
&\approx 2{,}92\ \mathrm{MPa}
\end{aligned}
$$

Le carré donne plus de poids à la grande erreur. Pour l’atelier de Camille, la cible reste binaire.